### SciBERT Model Architecture (Multilabel)

This notebook shows how `AutoModelForSequenceClassification` is configured for **SciBERT multilabel classification** with Hugging Face `Trainer`.

We will inspect:
1. Model/config setup (`AutoConfig`, label mappings, `problem_type`)
2. Classification head structure
3. Loss function used for multilabel (`BCEWithLogitsLoss`)
4. Optimizer + scheduler as created by `Trainer`


In [33]:
import yaml
import torch

from pathlib import Path
from pprint import pprint
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from arxiv_paper_discovery.label_taxonomy import (
    LABELS,
    LABEL_TO_IDX,
    IDX_TO_LABEL,
    NUM_CLASSES,
    labels_to_multihot,
    multihot_to_labels,
)

from arxiv_paper_discovery.config import CONFIG_DIR


In [13]:
# Load project config used by training script
cfg_path = CONFIG_DIR / 'scibert.yaml'
with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg

{'experiment': {'seed': 42},
 'model': {'pretrained_name': 'allenai/scibert_scivocab_uncased',
  'dropout_p': 0.1},
 'training': {'threshold': 0.35, 'sample_ratio': 0.135},
 'trainer': {'run_name': 'scibert_classification',
  'num_train_epochs': 3,
  'per_device_train_batch_size': 4,
  'per_device_eval_batch_size': 4,
  'gradient_accumulation_steps': 4,
  'learning_rate': 2e-05,
  'weight_decay': 0.01,
  'optim': 'adamw_torch',
  'lr_scheduler_type': 'linear',
  'warmup_ratio': 0.1,
  'logging_strategy': 'steps',
  'logging_steps': 50,
  'eval_strategy': 'steps',
  'eval_steps': 6000,
  'save_strategy': 'steps',
  'save_steps': 6000,
  'save_total_limit': 2,
  'load_best_model_at_end': True,
  'metric_for_best_model': 'eval_f1',
  'greater_is_better': True,
  'report_to': ['none'],
  'disable_tqdm': False,
  'fp16': True,
  'bf16': False},
 'sweep': {'trainer.learning_rate': [1e-05, 3e-05, 4e-05, 0.0001],
  'model.dropout_p': [0.1, 0.3, 0.5]}}

In [16]:
pretrained_name = cfg['model']['pretrained_name']
dropout_p = cfg['model'].get('dropout_p')


print('pretrained_name:', pretrained_name)
print('num_labels:', len(LABELS))
print('first labels:', LABELS[:5])

# Example conversion helpers from taxonomy
example_groups = ['Machine Learning', 'Quantum Physics']
example_vec = labels_to_multihot(example_groups)
print('multihot sum:', sum(example_vec))
print('decoded labels:', multihot_to_labels(example_vec))

pretrained_name: allenai/scibert_scivocab_uncased
num_labels: 25
first labels: ['Applied and Interdisciplinary Physics', 'Astrophysics', 'CS Theory and Algorithms', 'Computer Systems and Networking', 'Computer Vision']
multihot sum: 2
decoded labels: ['Machine Learning', 'Quantum Physics']


In [18]:
# Build SciBERT config for multilabel classification
model_config = AutoConfig.from_pretrained(
    pretrained_name,
    num_labels=NUM_CLASSES,
    problem_type='multi_label_classification',
    label2id=LABEL_TO_IDX,
    id2label=IDX_TO_LABEL,
)

if dropout_p is not None:
    model_config.classifier_dropout = float(dropout_p)

print('model_type:', model_config.model_type)
print('hidden_size:', model_config.hidden_size)
print('num_labels:', model_config.num_labels)
print('problem_type:', model_config.problem_type)
print('classifier_dropout:', getattr(model_config, 'classifier_dropout', None))


model_type: bert
hidden_size: 768
num_labels: 25
problem_type: multi_label_classification
classifier_dropout: 0.1


In [30]:
# Instantiate tokenizer + model
# This is the exact model family selected by your config:
# allenai/scibert_scivocab_uncased

tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
model = AutoModelForSequenceClassification.from_pretrained(pretrained_name, config=model_config)

model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29143.38it/s]
BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [29]:
# Inspect classifier head parameters
print('Classifier module:', model.classifier)

Classifier module: Linear(in_features=768, out_features=25, bias=True)
